# Search 

1. source: https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=8


2. Search Libraries:: lucene search, elastic search, solar, minsearch

3. for small dataset, we use minsearch

## get the documents (data set)

In [3]:
import requests

# get the available courses
# each doc has course, course_path, path, questions_count fields
# the path contians the post past, "https://datatalks.club/faq" + "/json/data-engineering-zoomcamp.json" is wher ethe doc answer/question is saved 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()



documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    # get a list of dictionaries from a specific course
    course_data = course_response.json()
    # extend the list
    documents.extend(course_data)

len(documents)

1342

In [4]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [6]:
documents[1000]

{'id': '947bf26d84',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 3. Machine Learning for Classification',
 'question': 'pandas.get_dummies() and DictVectorizer(sparse=False) produce the same type of one-hot encodings:',
 'answer': '<{IMAGE:image_1}>\n\n`DictVectorizer(sparse=True)` produces [CSR](https://en.wikipedia.org/wiki/Sparse_matrix#Compressed_sparse_row_(CSR,_CRS_or_Yale_format)) format, which is both more memory efficient and converges better during fit().\n\nIt stores non-zero values and indices instead of adding a column for each class of each feature, which can result in large numbers of columns (e.g., models of cars).\n\nUsing "sparse" format is slower (around 6-8 minutes for Q6 task - Linear/Ridge Regression) for a high number of classes (like car models) and produces slightly worse results in both Logistic and Linear/Ridge Regression.\n\nIt also generates convergence warnings for Linear/Ridge Regression.'}

In [14]:
from minsearch import Index

question = "I just discovered the course. Can I join now?"

index = Index(
    text_fields=["question", "section", "answer"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
    keyword_fields=["course"], # this is used when I want to filter, for example select * from documents where section="Machine Learning for Classification" then I will ignore all the courses and I will use only that section,
)
# fit with the documents (json file that contains all the prepared data)
index.fit(documents)

# then search (general search)
general_search = index.search(question)

# filter specifc course, to get the search according from specifc course for example, i.e. from keyword_fields
filtered_search = index.search(
                          question,
                          filter_dict={"course": "machine-learning-zoomcamp"},
                          num_results=5, # get the top 5 resuls, not all of them
                          boost_dict= {  # this for telling which field are important for example question is so important when there is a word in the question, then immediately gets the search from the question field
                                       "question":2, # this is so important field to look first from search
                                       "section": 0.5,
                                       "answer":1, # default value is 1 
                                    },
                          )

filtered_search

[{'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': 'e0a95572a6',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How do I join Slack if the invite email didn’t arrive?',
  'answer': 'Go to DataTalks.Club, request a Slack invite, or use the manual request form (processed daily). After joining, browse channels and join **#course-ml-zoomcamp**.'},
 {'

# put in a function

In [ ]:
from minsearch import Index



def search(question, course="llm-zoomcamp", num_results=5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

index = Index(
    text_fields=["question", "section", "answer"], # fields that conatin useful information to get the answer (fields that are used to perfom search)
    keyword_fields=["course"], # this is used when I want to filter, for example select * from documents where section="Machine Learning for Classification" then I will ignore all the courses and I will use only that section,
)
# fit with the documents (json file that contains all the prepared data)
index.fit(documents)

# get search result
question = "I just discovered the course. Can I join now?"
search_result = search(question, course="llm-zoomcamp", num_results=5)